In [ ]:
import os
import random
import numpy as np
import requests
import json
import argparse
import time
prompts = [
    # 1.Philosophical Question:
    "What is the meaning of true happiness in life?",

    # 2.Hypothetical Scenario:
    "If humans could live on Mars, what challenges would they face and how could they overcome them?",

    # 3.Creative Thinking Prompt:
    "Can you describe an imaginary city where technology and nature exist in perfect harmony?",

    # 4.Practical Advice Question:
    "What are the most effective ways to learn a new language quickly?",

    # 5.Exploration of Abstract Concepts:
    "How would you explain the concept of time to someone who has never experienced it?",
    
    # 6.Scientific Exploration:
    "What are the possible effects of artificial intelligence on scientific research in the next decade?",

    # 7.Ethical Dilemma:
    "Is it ever justifiable to prioritize technological advancement over environmental protection?",

    # 8.Problem-Solving Question:
    "How can cities effectively reduce traffic congestion without compromising accessibility?",

    # 9.Imaginative Scenario:
    "If animals could communicate with humans, how would that change our world?",

    # 10.Personal Reflection Prompt:
    "What qualities make someone a great leader, and how can those qualities be developed?"
    ]


API_KEY = "" 
API_ENDPOINT = ""


def generate_outputs_api(model_name, prompt, num_samples, max_length, catch, max_retries=10):
    outputs = []
    generated_count = 0
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json",
    }

    while generated_count < num_samples:
        samples_to_generate = min(catch, num_samples - generated_count)
        retries = 0
        wait_time = 2  
 
        while retries < max_retries:
            data = {
                "model": model_name,
                "messages": [
                    {"role": "system", "content": (
                            "You are a helpful assistant.Your answers should be plain text "
                            "without using special characters, bullet points, or lists. "
                            "Don't repeat the question in your answer and just provide the answer "
                            "at most 125 tokens."
                        )},
                    {"role": "user", "content": prompt},
                ],
                "max_tokens": max_length,
                "temperature": 0.9,
                "top_p": 0.95,
                "n": samples_to_generate,
            }

            try:
                response = requests.post(API_ENDPOINT, json=data, headers=headers)
                response.raise_for_status()
                response_data = response.json()

                temp_outputs = [choice.get("message", {}).get("content", "").strip() for choice in response_data.get("choices", [])]
                
                if all(temp_outputs) and len(temp_outputs) == samples_to_generate:
                    outputs.extend(temp_outputs)
                    generated_count += len(temp_outputs)

                    if generated_count % 100 == 0:
                        print(f"Generated {generated_count} outputs so far...")

                    break  
                else:
                    print(f"Invalid response, retrying in {wait_time}s... ({retries+1}/{max_retries})")
                    retries += 1
                    time.sleep(wait_time)
                    wait_time *= 2

            except requests.exceptions.RequestException as e:
                print(f"API request error: {e}, retrying in {wait_time}s... ({retries+1}/{max_retries})")
                retries += 1
                time.sleep(wait_time)
                wait_time *= 2

        if retries == max_retries:
            print(f"Failed to generate valid response after {max_retries} attempts. Skipping this batch.")

    return outputs


model_names = ["gpt-4o"]
num_samples = [2000]
max_length = 125

for model_name in model_names:
    for num_sample in num_samples:
        output_file = f"{model_name}_{num_sample}{max_length}.json"
        all_outputs = {}


        if os.path.exists(output_file):
            with open(output_file, "r") as f:
                all_outputs = json.load(f)
                print(f"Loaded existing file: {output_file}. Continuing generation for missing prompts.")

        for prompt in prompts:
            if prompt in all_outputs:
                print(f"Prompt already processed: {prompt}. Skipping.")
                continue

        
            outputs = generate_outputs_api(model_name, prompt, num_samples=num_sample, max_length=max_length, catch=50)
            print(f"Prompt: {prompt} | Total responses generated: {len(outputs)}")
            all_outputs[prompt] = outputs


            with open(output_file, "w") as f:
                json.dump(all_outputs, f, indent=4)
            
            print(f"Progress saved for prompt: {prompt}. File updated: {output_file}")

        print(f"Generation completed for {model_name}. Full results saved to {output_file}.")

Generated 100 outputs so far...
Generated 200 outputs so far...
Generated 300 outputs so far...
Generated 400 outputs so far...
Generated 500 outputs so far...
Generated 600 outputs so far...
Generated 700 outputs so far...
Generated 800 outputs so far...
Generated 900 outputs so far...
Generated 1000 outputs so far...
Generated 1100 outputs so far...
Generated 1200 outputs so far...
Generated 1300 outputs so far...
Generated 1400 outputs so far...
Generated 1500 outputs so far...
Generated 1600 outputs so far...
Generated 1700 outputs so far...
Generated 1800 outputs so far...
Generated 1900 outputs so far...
Generated 2000 outputs so far...
Prompt: What is the meaning of true happiness in life? | Total responses generated: 2000
Progress saved for prompt: What is the meaning of true happiness in life?. File updated: gpt-4o_2000125.json
Generated 100 outputs so far...
Generated 200 outputs so far...
Generated 300 outputs so far...
Generated 400 outputs so far...
Generated 500 outputs s